# Nonlinear Optimizer Case Studies

Focused runs for selected chapter-2-style instances, comparing restarts and initialization effects while saving outputs under `results/optimizer_benchmarks/case_studies/`.


In [1]:
from pathlib import Path
import pandas as pd

from mpmgame.benchmark_suite import benchmark_problem_registry, run_benchmark_suite

RESULTS_DIR = Path('../results/optimizer_benchmarks/case_studies').resolve()
REPORTS_DIR = Path('../reports/optimizer_benchmarks/case_studies').resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print('results:', RESULTS_DIR)
print('reports:', REPORTS_DIR)


results: C:\Users\perry\Documents\VSCode\masked-perturbation-model\results\optimizer_benchmarks\case_studies
reports: C:\Users\perry\Documents\VSCode\masked-perturbation-model\reports\optimizer_benchmarks\case_studies


C:\Users\perry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Case study A: Full attacker on 3x3 toy


In [2]:
case_a = [p for p in benchmark_problem_registry('full') if p.problem_id == 'toy3_w1_full']
algos_a = ['Nelder-Mead', 'Powell', 'SLSQP', 'trust-constr', 'differential_evolution', 'basinhopping']

raw_a, summary_a = run_benchmark_suite(
    problem_specs=case_a,
    algorithms=algos_a,
    n_restarts=10,
    timeout_per_run_sec=60,
    algorithm_timeout_sec=12 * 60,
    output_dir=RESULTS_DIR / 'case_a',
    report_dir=REPORTS_DIR / 'case_a',
    seed=101,
    show_progress=True,
    maxiter=300,
)
display(summary_a.sort_values('best_feasible_objective'))


Problems:   0%|          | 0/1 [00:00<?, ?it/s]



































































































































Problems: 100%|██████████| 1/1 [03:52<00:00, 232.20s/it]


,problem_id,algorithm,runs,best_feasible_objective,median_feasible_objective,fraction_feasible,fraction_timeout,mean_runtime_sec,best_runtime_to_feasible,robustness_score
0,toy3_w1_full,Nelder-Mead,10,0.706888,0.706888,1.0,0.0,0.030310,0.010774,8.757585e-01
6,toy3_w1_full,basinhopping,10,0.706888,0.706888,1.0,0.0,0.750487,0.334317,8.757585e-01
7,toy3_w1_full,differential_evolution,10,0.706888,0.706888,1.0,0.0,0.160598,0.068614,8.757585e-01
1,toy3_w1_full,Powell,10,0.706888,0.706888,1.0,0.0,0.058551,0.034838,8.757585e-01
2,toy3_w1_full,SLSQP,10,0.706888,0.706888,1.0,0.0,0.006713,0.003088,8.757585e-01
8,toy3_w1_full,trust-constr,10,0.706888,0.706888,1.0,0.0,0.011980,0.006814,8.757585e-01
4,toy3_w1_full,baseline_projection,1,0.706888,0.706888,1.0,0.0,0.021369,0.021369,8.757585e-01
5,toy3_w1_full,baseline_zero,1,0.706888,0.706888,1.0,0.0,0.000000,0.000000,8.757585e-01
3,toy3_w1_full,baseline_lp,1,inf,inf,0.0,0.0,0.046946,inf,3.000000e-07


## Case study B: Single-link attacker on 3x3 toy


In [3]:
case_b = [p for p in benchmark_problem_registry('quick') if p.problem_id == 'toy3_w2_single']
algos_b = ['COBYLA', 'SLSQP', 'L-BFGS-B', 'dual_annealing', 'shgo']

raw_b, summary_b = run_benchmark_suite(
    problem_specs=case_b,
    algorithms=algos_b,
    n_restarts=8,
    timeout_per_run_sec=45,
    output_dir=RESULTS_DIR / 'case_b',
    report_dir=REPORTS_DIR / 'case_b',
    seed=202,
    show_progress=True,
    maxiter=250,
)
display(summary_b.sort_values('best_feasible_objective'))


Problems:   0%|          | 0/1 [00:00<?, ?it/s]

























































































Problems: 100%|██████████| 1/1 [05:02<00:00, 302.48s/it]


,problem_id,algorithm,runs,best_feasible_objective,median_feasible_objective,fraction_feasible,fraction_timeout,mean_runtime_sec,best_runtime_to_feasible,robustness_score
6,toy3_w2_single,dual_annealing,8,0.897716,0.902556,1.00,0.0,3.616418,2.221256,8.580848e-01
7,toy3_w2_single,shgo,8,0.897716,0.897716,1.00,0.0,15.225224,10.228299,8.580848e-01
1,toy3_w2_single,L-BFGS-B,8,0.897716,0.904445,1.00,0.0,0.389834,0.096192,8.580848e-01
2,toy3_w2_single,SLSQP,8,0.897716,0.904445,1.00,0.0,0.125692,0.029143,8.580848e-01
0,toy3_w2_single,COBYLA,8,0.920429,0.941638,0.75,0.0,0.082353,0.028909,6.812151e-01
5,toy3_w2_single,baseline_zero,1,1.003034,1.003034,1.00,0.0,0.000000,0.000000,8.497728e-01
3,toy3_w2_single,baseline_lp,1,inf,inf,0.00,0.0,0.026349,inf,3.000000e-07
4,toy3_w2_single,baseline_projection,1,inf,inf,0.00,0.0,0.083969,inf,3.000000e-07


## Cross-case summary


In [4]:
def best_by_algo(summary_df, label):
    out = summary_df[['algorithm', 'best_feasible_objective', 'fraction_feasible', 'mean_runtime_sec']].copy()
    out['case'] = label
    return out

combined = pd.concat([best_by_algo(summary_a, 'A'), best_by_algo(summary_b, 'B')], ignore_index=True)
display(combined.sort_values(['case', 'best_feasible_objective']))


,algorithm,best_feasible_objective,fraction_feasible,mean_runtime_sec,case
0,Nelder-Mead,0.706888,1.00,0.030310,A
1,basinhopping,0.706888,1.00,0.750487,A
2,differential_evolution,0.706888,1.00,0.160598,A
3,Powell,0.706888,1.00,0.058551,A
4,SLSQP,0.706888,1.00,0.006713,A
5,trust-constr,0.706888,1.00,0.011980,A
6,baseline_projection,0.706888,1.00,0.021369,A
7,baseline_zero,0.706888,1.00,0.000000,A
8,baseline_lp,inf,0.00,0.046946,A
9,dual_annealing,0.897716,1.00,3.616418,B


In [5]:
print('Saved case-study files:')
for f in sorted(RESULTS_DIR.rglob('*')):
    if f.is_file():
        print(' ', f.relative_to(RESULTS_DIR))
for f in sorted(REPORTS_DIR.rglob('*')):
    if f.is_file():
        print(' ', Path('reports') / f.relative_to(REPORTS_DIR.parent))


Saved case-study files:
  case_a\benchmark_raw_results.csv
  case_a\benchmark_summary.csv
  case_a\run_config.json
  case_b\benchmark_raw_results.csv
  case_b\benchmark_summary.csv
  case_b\run_config.json
  reports\case_studies\case_a\baseline_comparison.png
  reports\case_studies\case_a\benchmark_report.md
  reports\case_studies\case_a\best_feasible_bar.png
  reports\case_studies\case_a\best_q_heatmap.png
  reports\case_studies\case_a\convergence_traces.png
  reports\case_studies\case_a\feasibility_rate.png
  reports\case_studies\case_a\objective_boxplot.png
  reports\case_studies\case_a\runtime_vs_objective.png
  reports\case_studies\case_a\timeout_count.png
  reports\case_studies\case_b\baseline_comparison.png
  reports\case_studies\case_b\benchmark_report.md
  reports\case_studies\case_b\best_feasible_bar.png
  reports\case_studies\case_b\best_q_heatmap.png
  reports\case_studies\case_b\convergence_traces.png
  reports\case_studies\case_b\feasibility_rate.png
  reports\case_studie